# Laboratorio de regresión - 5

|                |   |
:----------------|---|
| **Nombre**     |  Alexa Bojoruqez Valdivia |
| **Fecha**      |  14 de octubre |
| **Expediente** |  745497 |

## Validación

Hemos estado usando `train_test_split` en nuestros modelos anteriores.

¿Por qué?

- Utilizamos train_test_split para separar una parte de los datos para entrenar el modelo y otra para probarlo. Así podemos evaluar qué tan bien funciona el modelo con datos que no utilizó durante el entrenamiento.

Si la muestra es un subset de la población y queremos generalizar sobre la población, ¿no sería mejor utilizar todos los datos al entrenar un modelo?

- Utilizar todos los datos puede ayudar a entrenar mejor el modelo, pero entonces no tendríamos datos diferentes para comprobar si puede generalizar. Por eso separamos una parte para validarlo y saber cómo podría comportarse con observaciones nuevas.

El propósito de volver a muestrear dentro de nuestro dataset es tener una idea de qué tan buena podría ser la generalización de nuestro modelo. Imagina un dataset ya separado en dos mitades. Utilizas la primera mitad para entrenar el modelo y pruebas en la segunda mitad; la segunda mitad eran datos invisibles para el modelo al momento de entrenar. Esto nos lleva a tres escenario típicos:

1. Si el modelo hace buenas predicciones en la segunda mitad, significa que la primera mitad era "suficiente" para generalizar.
2. Si el modelo no hace buenas predicciones en la segunda mitad, pero sí en la primera mitad, podría ser que había información importante en la segunda mitad que debió haber sido tomada en cuenta al entrenar, o un problema de overfitting.
3. Si el modelo no hace buenas predicciones en la segunda mitad, y tampoco en la primera mitad, se tendrían que revisar los factores y/o el modelo seleccionado.

El caso ideal sería el 1, pero por estadística los errores y varianzas tienen como entrada el número de muestas, por lo que tenemos menos seguridad de nuestros resutados al usar menos muestras. Si vemos que el modelo generaliza bien podemos unir de nuevo el dataset y entrenar sobre el dataset completo.

En el caso 2 está el problema de que no podemos saber qué información es necesaria para el entrenamiento apropiado del modelo; esto nos lleva a pensar que debemos usar el dataset completo para entrenar, pero esto nos lleva al mismo problema de no saber si el modelo puede generalizar.

El problema sólo incrementa si se tienen hiperparámetros en el modelo (e.g. $\lambda$ en regularización).

## Leave-One-Out Cross Validation

Este método de validación es una colección de $n$ `train-test-split`. Teniendo un dataset de $n$ muestras, la lógica es:
1. Saca una muestra del dataset.
2. Entrena tu modelo con las $n-1$ muestras.
3. Evalúa tu modelo en la muestra que quedó fuera con el métrico que más se ajuste a la aplicación.
4. Regresa la muestra al dataset.
5. Repite 1-4 con muestras diferentes hasta haber hecho el procedimiento $n$ veces para $n$ muestras.
6. Calcula la media y desviación estándar de los métricos guardados.

Con los resultados del proceso de validación podemos saber qué tan bueno podría ser el modelo seleccionado con los datos (con/sin transformaciones).

### Ejercicio 1

Utiliza el dataset `Motor Trend Car Road Tests`. Elimina la columna `model` y entrena 32 modelos diferentes utilizando Leave-One-Out Cross Validation con target `mpg`. Utiliza MSE como métrico.

In [2]:
import pandas as pd

data = pd.read_excel("Motor Trend Car Road Tests.xlsx")

data.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [3]:
data = data.drop(columns=['model'])

data.head()

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [4]:
X = data.drop(columns=['mpg'])
y = data['mpg']

In [5]:
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()

In [6]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()

In [9]:
from sklearn.model_selection import cross_val_score

mse = cross_val_score(
    modelo,
    X,
    y,
    cv=loo,
    scoring='neg_mean_squared_error'
)

print(mse)

[-5.25885580e+00 -2.45399671e+00 -2.05504934e+01 -4.43291071e-02
 -1.58115552e+00 -1.01186255e+01 -1.63740415e-02 -8.08125071e+00
 -3.94701835e+01 -7.70632772e-01 -4.95569423e+00 -1.02247302e+01
 -4.43012933e+00 -4.87812771e-01 -6.82384458e+00 -6.02750698e-01
 -3.68101786e+01 -3.17598556e+01 -1.06322754e+00 -3.27132862e+01
 -1.43071737e+01 -3.40535465e+00 -9.40800765e+00 -1.03500398e-04
 -9.96243953e+00 -1.34122089e+00 -1.64800641e-01 -2.36002065e+01
 -8.31145764e+01 -1.02712372e-04 -8.78553337e+00 -1.75029299e+01]


In [10]:
import numpy as np

print("MSE promedio:", np.mean(mse))
print("Desviación estándar:", np.std(mse))

MSE promedio: -12.181558006901959
Desviación estándar: 17.06739987188857


Se entrenaron 32 modelos utilizando LOOCV, dejando una observación diferente fuera en cada modelo. El MSE promedio fue de -12.1816, lo que representa el error cuadrático promedio de las predicciones. La desviación estándar fue de 17.0674, que indica qué tanto varió el error entre las diferentes observaciones.

## K-Folds Cross-Validation

El dataset `Motor Trend Car Road Tests` sólo tiene 32 muestras, y utilizar un modelo sencillo de regresión múltiple hace que usar LOOCV sea muy rápido. El dataset `California Housing` tiene $20640$ muestras para $9$ columnas, entonces realizar un ajuste sobre una transformación o sobre el modelo y luego calcular el impacto esperado podría tomar más tiempo.

La solución propuesta es dividir el dataset en *k* folds (partes iguales), ajustar en *k-1* folds y probar en el restante.

### Ejercicio 2
Utiliza el dataset `California Housing` y haz K-folds Cross Validation con 10 folds. Utiliza el MSE como métrico.

In [11]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print("Dataset Shape:", housing.data.shape, housing.target.shape)
print("Dataset Features:", housing.feature_names)
print("Dataset Target:", housing.target_names)
X = housing.data
y = housing.target

Dataset Shape: (20640, 8) (20640,)
Dataset Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Dataset Target: ['MedHouseVal']


In [12]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=10)

In [13]:
mse_kfold = cross_val_score(
    modelo,
    X,
    y,
    cv=kf,
    scoring='neg_mean_squared_error'
)

In [14]:
mse_kfold = -mse_kfold
print(mse_kfold)

[0.48922052 0.43335865 0.8864377  0.39091641 0.7479731  0.52980278
 0.28798456 0.77326441 0.64305557 0.3275106 ]


In [15]:
print("MSE promedio:", np.mean(mse_kfold))
print("Desviación estándar:", np.std(mse_kfold))

MSE promedio: 0.5509524296956656
Desviación estándar: 0.19288582953865094


Interpreta.

- El modelo obtuvo un MSE promedio de 0.551 en los 10 folds, lo que representa el error cuadrático promedio de las predicciones. La desviación estándar fue de 0.193, por lo que el error cambia entre los diferentes folds, pero se mantiene relativamente cercano al promedio.

## Referencia

James, G., Witten, D., Hastie, T., Tibshirani, R.,, Taylor, J. (2023). An Introduction to Statistical Learning with Applications in Python. Cham: Springer. ISBN: 978-3-031-38746-3